# Vul 原始数据集格式化

对 `data/raw/vul/` 下的两个原始数据集的 `vul_code` 字段做**一次性原地格式化**，
使后续所有依赖这些文件的脚本无需重复格式化。

- `data/raw/vul/BigVul.jsonl`（JSONL 格式）
- `data/raw/vul/PrimeVul.jsonl`（JSONL 格式）

**幂等保证**：格式化完成后写入标记文件 `data/raw/vul/.formatted`，再次运行时自动跳过。

In [1]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.utils.formatter import auto_format_c_code_batch

VUL_RAW_DIR    = PROJECT_ROOT / "data/raw/vul"
FORMATTED_FLAG = VUL_RAW_DIR / ".formatted"

if FORMATTED_FLAG.exists():
    print("[SKIP] 标记文件已存在，Vul 原始数据集已格式化，跳过。")
else:
    print("开始格式化 Vul 原始数据集...")

PROJECT_ROOT: <PROJECT_ROOT>
开始格式化 Vul 原始数据集...


In [2]:
# ========== 通用 JSONL 格式化函数 ==========
def format_vul_jsonl(path: Path):
    """读取 JSONL 文件，对 vul_code 字段格式化后原地写回。"""
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"{path.name}: 读取 {len(records)} 条")

    codes = [r["vul_code"] for r in records]
    print(f"  格式化 {len(codes)} 条 vul_code...")
    results = auto_format_c_code_batch(codes, on_error="keep")
    for record, (formatted, _) in zip(records, results):
        record["vul_code"] = formatted

    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"  ✅ {path.name} 格式化完成，已原地写回 → {path}")

In [3]:
# ========== 格式化 BigVul ==========
if not FORMATTED_FLAG.exists():
    format_vul_jsonl(VUL_RAW_DIR / "BigVul.jsonl")
else:
    print("[SKIP] BigVul")

BigVul.jsonl: 读取 8763 条
  格式化 8763 条 vul_code...


<CONDA_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Formatting: 100%|██████████| 8763/8763 [01:11<00:00, 122.00snippet/s]


  ✅ BigVul.jsonl 格式化完成，已原地写回 → <PROJECT_ROOT>/data/raw/vul/BigVul.jsonl


In [4]:
# ========== 格式化 PrimeVul ==========
if not FORMATTED_FLAG.exists():
    format_vul_jsonl(VUL_RAW_DIR / "PrimeVul.jsonl")
else:
    print("[SKIP] PrimeVul")

PrimeVul.jsonl: 读取 5969 条
  格式化 5969 条 vul_code...


Formatting: 100%|██████████| 5969/5969 [00:47<00:00, 125.03snippet/s]


  ✅ PrimeVul.jsonl 格式化完成，已原地写回 → <PROJECT_ROOT>/data/raw/vul/PrimeVul.jsonl


In [5]:
# ========== 写入完成标记 ==========
if not FORMATTED_FLAG.exists():
    FORMATTED_FLAG.touch()
    print(f"✅ 所有 Vul 原始数据集格式化完成，标记已写入 → {FORMATTED_FLAG}")
else:
    print("[SKIP] 标记文件已存在，无需重复格式化。")

✅ 所有 Vul 原始数据集格式化完成，标记已写入 → <PROJECT_ROOT>/data/raw/vul/.formatted
